# Module 0, Notebook 1: The Agent-Environment Interaction Loop

## Learning Objectives
By completing this notebook, you will:
1. Understand the agent-environment loop as the fundamental abstraction in reinforcement learning
2. Build a GridWorld environment from scratch using only NumPy
3. Implement a random agent and a greedy agent, and compare their behavior
4. Visualize agent trajectories to build intuition about exploration vs. exploitation
5. Measure and plot cumulative rewards to quantify agent performance

## Prerequisites
- Basic Python (classes, functions, loops)
- NumPy array indexing and operations
- Matplotlib for plotting

## Estimated Time: 15 minutes

---

**Key Insight:** Every reinforcement learning algorithm, from the simplest random walker to the most sophisticated deep RL system, is built on one loop: the agent observes a state, selects an action, the environment transitions to a new state and emits a reward, repeat. Getting this loop right is the foundation of everything that follows.

In [ ]:
learning_objectives([
    "Basic Python (classes, functions, loops)",
    "NumPy array indexing and operations",
    "Matplotlib for plotting",
    "**Key Insight:** Every reinforcement learning algorithm, from the simplest random walker to the most sophisticated deep RL system, is built on one loop: the agent observes a state, selects an action, the environment transitions to a new state and emits a reward, repeat. Getting this loop right is the foundation of everything that follows."
])

## 1. Setup

We use only NumPy and Matplotlib throughout this notebook. The seed ensures your results match the expected outputs.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Import dependencies and fix the random seed for reproducibility
# Key Concept: Reproducibility is essential when debugging stochastic systems

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

np.random.seed(42)

print("Dependencies loaded. NumPy version:", np.__version__)

## 2. The Agent-Environment Interface

In reinforcement learning, two entities interact:

- **Environment**: holds the true state of the world, applies actions, emits observations and rewards
- **Agent**: receives observations, selects actions, collects rewards

The standard interface has three methods:

| Method | Input | Output | Who calls it |
|--------|-------|--------|-------------|
| `reset()` | — | initial state | agent (start of episode) |
| `step(action)` | action | (next_state, reward, done) | agent (each timestep) |
| `render()` | — | visual | optional |

This interface is intentionally minimal. The agent does not see the environment's internals — it only sees observations and rewards. This separation is not just good software design; it matches how real-world systems work.

## 3. Building GridWorld from Scratch

GridWorld is the canonical RL teaching environment. The agent moves on a 2D grid, collecting rewards and avoiding penalties. We implement it as a Python class with the standard reset/step interface.

**Grid layout:**
- `S` = start position (top-left)
- `G` = goal (bottom-right), reward = +10
- `H` = hole (penalty), reward = -5, episode ends
- `.` = empty cell, reward = -0.1 (small step cost to encourage efficiency)

The step cost of -0.1 is a deliberate design choice: without it, the agent has no incentive to reach the goal quickly.

In [ ]:
# Purpose: Define the GridWorld environment
# Key Concept: The environment encapsulates state transitions and reward logic

class GridWorld:
    """
    A simple 4x4 GridWorld environment.

    Grid legend:
        0 = empty cell  (reward: -0.1)
        1 = hole        (reward: -5.0, episode ends)
        2 = goal        (reward: +10.0, episode ends)
        3 = start       (treated as empty during play)

    Actions: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT
    """

    ACTION_UP    = 0
    ACTION_DOWN  = 1
    ACTION_LEFT  = 2
    ACTION_RIGHT = 3
    N_ACTIONS    = 4

    # Human-readable action names used in visualizations
    ACTION_SYMBOLS = {0: '↑', 1: '↓', 2: '←', 3: '→'}

    def __init__(self, grid=None):
        """
        Parameters
        ----------
        grid : np.ndarray, optional
            4x4 integer array. If None, uses default layout.
        """
        if grid is None:
            # Default 4x4 layout: 0=empty, 1=hole, 2=goal, 3=start
            self.grid = np.array([
                [3, 0, 0, 0],
                [0, 1, 0, 1],
                [0, 0, 0, 1],
                [1, 0, 0, 2]
            ])
        else:
            self.grid = grid

        self.n_rows, self.n_cols = self.grid.shape
        self.n_states = self.n_rows * self.n_cols

        # Find start position from grid
        start_coords = np.argwhere(self.grid == 3)
        self.start_pos = tuple(start_coords[0]) if len(start_coords) > 0 else (0, 0)

        # Reward structure
        self.rewards = {0: -0.1, 1: -5.0, 2: 10.0, 3: -0.1}

        self.agent_pos = self.start_pos
        self.done = False

    def reset(self):
        """Reset environment to start. Returns initial state index."""
        self.agent_pos = self.start_pos
        self.done = False
        return self._pos_to_state(self.agent_pos)

    def step(self, action):
        """
        Execute one action.

        Parameters
        ----------
        action : int
            0=UP, 1=DOWN, 2=LEFT, 3=RIGHT

        Returns
        -------
        next_state : int
            Flat state index
        reward : float
        done : bool
        info : dict
            Contains 'pos' (row, col) for trajectory logging
        """
        if self.done:
            raise RuntimeError("Call reset() before stepping after episode ends.")

        row, col = self.agent_pos

        # Compute candidate next position
        delta = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
        dr, dc = delta[action]
        new_row = row + dr
        new_col = col + dc

        # Wall collision: stay in place (still costs the step penalty)
        if 0 <= new_row < self.n_rows and 0 <= new_col < self.n_cols:
            self.agent_pos = (new_row, new_col)

        # Look up cell type and reward
        cell_type = self.grid[self.agent_pos]
        reward = self.rewards[cell_type]

        # Episode ends on hole (1) or goal (2)
        self.done = cell_type in (1, 2)

        next_state = self._pos_to_state(self.agent_pos)
        return next_state, reward, self.done, {'pos': self.agent_pos}

    def _pos_to_state(self, pos):
        """Convert (row, col) to flat integer state index."""
        return pos[0] * self.n_cols + pos[1]

    def _state_to_pos(self, state):
        """Convert flat state index to (row, col)."""
        return (state // self.n_cols, state % self.n_cols)

    def render(self):
        """Print a text representation of the current grid state."""
        symbols = {0: '.', 1: 'H', 2: 'G', 3: 'S'}
        print("\n+" + "---+" * self.n_cols)
        for r in range(self.n_rows):
            row_str = "|"
            for c in range(self.n_cols):
                if (r, c) == self.agent_pos:
                    row_str += " A |"
                else:
                    row_str += f" {symbols[self.grid[r, c]]} |"
            print(row_str)
            print("+" + "---+" * self.n_cols)


# Smoke test: reset and take one step
env = GridWorld()
state = env.reset()
print(f"Initial state index: {state}  (position: {env.agent_pos})")
next_state, reward, done, info = env.step(GridWorld.ACTION_DOWN)
print(f"After DOWN: state={next_state}, reward={reward}, done={done}, pos={info['pos']}")
env.render()

## 4. Implementing Agents

We implement two agents to contrast their behavior:

1. **RandomAgent**: selects actions uniformly at random — a baseline that ignores all information
2. **GreedyAgent**: maintains a Q-table and always selects the action with the highest estimated value — no exploration

Both agents share the same interaction loop. The only difference is how `select_action()` works. This separation of agent logic from environment logic is a key design principle.

In [ ]:
# Purpose: Define random and greedy agents
# Key Concept: Agents differ only in their action selection policy

class RandomAgent:
    """
    Selects actions uniformly at random.
    This is the simplest possible agent and serves as a performance baseline.
    """

    def __init__(self, n_actions):
        self.n_actions = n_actions

    def select_action(self, state):
        """Ignore state entirely; pick any action with equal probability."""
        return np.random.randint(self.n_actions)

    def update(self, state, action, reward, next_state, done):
        """Random agent does not learn; update is a no-op."""
        pass


class GreedyAgent:
    """
    Maintains a Q-table and always selects the highest-value action.

    The Q-table is updated using a simple average of observed returns.
    This agent exploits its current knowledge with no exploration.
    """

    def __init__(self, n_states, n_actions, alpha=0.1):
        """
        Parameters
        ----------
        n_states : int
        n_actions : int
        alpha : float
            Learning rate for Q-table updates.
        """
        self.n_actions = n_actions
        self.alpha = alpha
        # Initialize Q-values to zero (optimistic initialization would use positive values)
        self.Q = np.zeros((n_states, n_actions))

    def select_action(self, state):
        """Return action with highest Q-value; break ties randomly."""
        max_q = np.max(self.Q[state])
        # argmax with random tie-breaking avoids systematic bias
        best_actions = np.where(self.Q[state] == max_q)[0]
        return np.random.choice(best_actions)

    def update(self, state, action, reward, next_state, done):
        """
        One-step Q-learning update.
        Q(s,a) <- Q(s,a) + alpha * (reward + max_a' Q(s',a') - Q(s,a))
        """
        best_next = 0.0 if done else np.max(self.Q[next_state])
        td_target = reward + best_next
        self.Q[state, action] += self.alpha * (td_target - self.Q[state, action])


print("RandomAgent and GreedyAgent defined.")
print(f"Q-table shape for 16-state, 4-action world: {GreedyAgent(16, 4).Q.shape}")

## 5. The Interaction Loop

The interaction loop is the heartbeat of every RL system. Each iteration:
1. Agent observes current state `s`
2. Agent selects action `a` based on its policy
3. Environment transitions to `s'` and emits reward `r`
4. Agent updates its internal model (if it learns)
5. If episode is done, reset and start again

We run both agents for 200 episodes and record their per-episode returns.

In [ ]:
# Purpose: Define and run the agent-environment interaction loop
# Key Concept: The loop is identical for all agents; only select_action and update differ

def run_episodes(env, agent, n_episodes=200, max_steps=50):
    """
    Run the agent-environment loop for a fixed number of episodes.

    Parameters
    ----------
    env : GridWorld
    agent : RandomAgent or GreedyAgent
    n_episodes : int
    max_steps : int
        Cap steps per episode to prevent infinite loops.

    Returns
    -------
    episode_returns : list of float
        Total undiscounted reward per episode.
    last_trajectory : list of (row, col)
        Position sequence from the final episode (for visualization).
    """
    episode_returns = []
    last_trajectory = []

    for episode in range(n_episodes):
        state = env.reset()
        episode_return = 0.0
        trajectory = [env.agent_pos]  # Record visited positions

        for step in range(max_steps):
            # Step 1: Agent selects action
            action = agent.select_action(state)

            # Step 2: Environment transitions
            next_state, reward, done, info = env.step(action)

            # Step 3: Agent learns from the transition
            agent.update(state, action, reward, next_state, done)

            # Step 4: Accumulate reward and advance state
            episode_return += reward
            state = next_state
            trajectory.append(info['pos'])

            if done:
                break

        episode_returns.append(episode_return)
        last_trajectory = trajectory

    return episode_returns, last_trajectory


# Create environments and agents
env_random = GridWorld()
env_greedy = GridWorld()

random_agent = RandomAgent(n_actions=GridWorld.N_ACTIONS)
greedy_agent = GreedyAgent(n_states=16, n_actions=GridWorld.N_ACTIONS, alpha=0.1)

# Run episodes
N_EPISODES = 300
random_returns, random_traj = run_episodes(env_random, random_agent, N_EPISODES)
greedy_returns, greedy_traj = run_episodes(env_greedy, greedy_agent, N_EPISODES)

print(f"Random agent — mean return (last 50 eps): {np.mean(random_returns[-50:]):.2f}")
print(f"Greedy agent — mean return (last 50 eps): {np.mean(greedy_returns[-50:]):.2f}")

## 6. Visualizing Performance and Trajectories

Two plots reveal the difference between the agents:
1. **Learning curves**: cumulative reward over episodes (smoothed with a rolling window)
2. **Trajectory plots**: the path each agent took on its final episode

The greedy agent should show improvement over time; the random agent should stay flat.

In [ ]:
# Purpose: Visualize learning curves and final-episode trajectories side-by-side
# Key Concept: Smoothed returns reveal learning trends that raw episode rewards obscure

def smooth(values, window=20):
    """Rolling mean smoothing for noisy reward curves."""
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode='valid')


def plot_trajectory_on_grid(ax, grid, trajectory, title):
    """
    Draw the GridWorld and overlay an agent trajectory.

    Parameters
    ----------
    ax : matplotlib Axes
    grid : np.ndarray
    trajectory : list of (row, col)
    title : str
    """
    n_rows, n_cols = grid.shape

    # Color map: empty=white, hole=red, goal=green, start=blue
    color_map = {0: 'white', 1: '#ff6b6b', 2: '#51cf66', 3: '#74c0fc'}

    for r in range(n_rows):
        for c in range(n_cols):
            cell_type = grid[r, c]
            rect = mpatches.FancyBboxPatch(
                (c, n_rows - 1 - r), 1, 1,
                boxstyle="round,pad=0.05",
                facecolor=color_map[cell_type],
                edgecolor='#333333', linewidth=1.5
            )
            ax.add_patch(rect)
            label = {0: '', 1: 'H', 2: 'G', 3: 'S'}[cell_type]
            if label:
                ax.text(c + 0.5, n_rows - 0.5 - r, label,
                        ha='center', va='center', fontsize=12, fontweight='bold')

    # Draw trajectory as connected arrows
    if len(trajectory) > 1:
        traj_r = [n_rows - 0.5 - pos[0] for pos in trajectory]
        traj_c = [pos[1] + 0.5 for pos in trajectory]
        ax.plot(traj_c, traj_r, 'o-', color='#1c7ed6', linewidth=2,
                markersize=6, zorder=5, label='Path')
        # Mark start and end of trajectory
        ax.plot(traj_c[0], traj_r[0], 's', color='#1c7ed6', markersize=10, zorder=6)
        ax.plot(traj_c[-1], traj_r[-1], '*', color='gold', markersize=14, zorder=6)

    ax.set_xlim(0, n_cols)
    ax.set_ylim(0, n_rows)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')


# Build the figure
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Learning curves ---
ax = axes[0]
episodes = np.arange(len(smooth(random_returns)))
ax.plot(episodes, smooth(random_returns), color='#e64980', linewidth=2, label='Random agent')
ax.plot(episodes, smooth(greedy_returns), color='#1c7ed6', linewidth=2, label='Greedy agent')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Return (smoothed, window=20)', fontsize=11)
ax.set_title('Learning Curves', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

# --- Plot 2: Random agent trajectory ---
plot_trajectory_on_grid(axes[1], env_random.grid, random_traj,
                        f'Random Agent — Final Episode\n({len(random_traj)-1} steps)')

# --- Plot 3: Greedy agent trajectory ---
plot_trajectory_on_grid(axes[2], env_greedy.grid, greedy_traj,
                        f'Greedy Agent — Final Episode\n({len(greedy_traj)-1} steps)')

plt.tight_layout()
plt.savefig('agent_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to agent_comparison.png")

## 7. Exercises

### Exercise 1.1: Modify the Grid Layout

**Task:** Create a new 4x4 grid with a different layout. Add at least two more holes and move the goal to a different corner. Run both agents on your new grid for 300 episodes and compare their mean returns over the last 50 episodes.

**Expected Output:** Two mean return values, one per agent, printed to the console.

**Hints:**
<details>
<summary>Hint 1</summary>
The grid array uses: 0=empty, 1=hole, 2=goal, 3=start. Every grid must have exactly one 3 and at least one 2.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
Try placing holes at positions (0,2), (1,0), (2,3), (3,1) and the goal at (3,3). Make sure there is still a path from start to goal.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Step 1: Define your custom grid as a 4x4 numpy array
custom_grid = None  # Replace with your array

# Step 2: Create environments with custom_grid
# env1 = GridWorld(grid=custom_grid)
# env2 = GridWorld(grid=custom_grid)

# Step 3: Create agents

# Step 4: Run episodes and compare

mean_random = None  # Replace with computed value
mean_greedy = None  # Replace with computed value

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_1():
    assert custom_grid is not None, "Define your custom_grid array!"
    assert isinstance(custom_grid, np.ndarray), "custom_grid must be a numpy array."
    assert custom_grid.shape == (4, 4), f"Expected shape (4,4), got {custom_grid.shape}"
    assert 3 in custom_grid, "Grid must contain a start cell (value=3)."
    assert 2 in custom_grid, "Grid must contain a goal cell (value=2)."
    assert np.sum(custom_grid == 1) >= 2, "Add at least 2 holes (value=1)."
    assert mean_random is not None, "Compute mean_random!"
    assert mean_greedy is not None, "Compute mean_greedy!"
    assert isinstance(mean_random, float), "mean_random should be a float."
    print(f"Exercise 1.1 passed! Random={mean_random:.2f}, Greedy={mean_greedy:.2f}")

test_exercise_1_1()

### Exercise 1.2: Implement an Epsilon-Greedy Agent

**Task:** The pure greedy agent never explores. Implement an `EpsilonGreedyAgent` that selects a random action with probability `epsilon` and the greedy action with probability `1 - epsilon`. Run it with `epsilon=0.1` and compare it to the pure greedy agent.

**Expected Output:** The epsilon-greedy agent should achieve comparable or better mean returns than pure greedy.

**Hints:**
<details>
<summary>Hint 1</summary>
Use `np.random.random()` which returns a float in [0, 1). If it is less than `epsilon`, explore (random action); otherwise exploit (greedy action).
</details>

<details>
<summary>Hint 2 (more specific)</summary>
Your `select_action` should look like:
```python
if np.random.random() < self.epsilon:
    return np.random.randint(self.n_actions)
else:
    # greedy selection from Q-table
```
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
class EpsilonGreedyAgent:
    """
    Balances exploration and exploitation via epsilon-greedy action selection.
    """

    def __init__(self, n_states, n_actions, alpha=0.1, epsilon=0.1):
        self.n_actions = n_actions
        self.alpha = alpha
        self.epsilon = epsilon
        self.Q = np.zeros((n_states, n_actions))

    def select_action(self, state):
        # Replace this with your epsilon-greedy implementation
        return None  # Remove this line

    def update(self, state, action, reward, next_state, done):
        # Copy from GreedyAgent — the update rule is identical
        pass


# Test your agent
env_eps = GridWorld()
eps_agent = EpsilonGreedyAgent(n_states=16, n_actions=4, alpha=0.1, epsilon=0.1)
eps_returns, eps_traj = run_episodes(env_eps, eps_agent, N_EPISODES)

eps_mean = None  # Replace: mean return of last 50 episodes

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_2():
    agent = EpsilonGreedyAgent(n_states=16, n_actions=4, epsilon=0.1)

    # Test that action is valid
    np.random.seed(0)
    action = agent.select_action(0)
    assert action is not None, "select_action must return an action."
    assert action in range(4), f"Action {action} is not in [0, 3]."

    # Test that exploration occurs: run 1000 selections from state 0 and check randomness
    np.random.seed(123)
    actions = [agent.select_action(0) for _ in range(1000)]
    action_counts = np.bincount(actions, minlength=4)
    # All actions should appear at least a few times if epsilon > 0
    assert all(c > 0 for c in action_counts), \
        "All actions should be selected at least once with epsilon=0.1 over 1000 steps."

    # Test update modifies Q
    agent.update(0, 1, 1.0, 5, False)
    assert agent.Q[0, 1] != 0.0, "update() must modify Q[state, action]."

    assert eps_mean is not None, "Compute eps_mean (mean return of last 50 episodes)."
    print(f"Exercise 1.2 passed! EpsilonGreedy mean return (last 50): {eps_mean:.2f}")

test_exercise_1_2()

In [ ]:
section_divider("Key Takeaways")

## Key Takeaways

1. **The agent-environment loop** is the universal abstraction in RL: observe state, select action, receive reward, repeat.
2. **Environments** expose a minimal interface (`reset`, `step`) and hide their internals from the agent.
3. **The random agent** is not just a toy — it is the baseline every other agent must beat.
4. **Pure greedy agents** can get stuck exploiting suboptimal actions because they never explore.
5. **Epsilon-greedy** is the simplest fix: add a small probability of random action to ensure all state-action pairs are eventually visited.
6. **Trajectories and learning curves** are complementary views — one shows what happened, the other shows whether the agent is improving.

## What's Next

In Notebook 2 (`02_mdp_builder.ipynb`), we formalize the interaction loop mathematically as a **Markov Decision Process**. We will define transition probabilities, compute returns explicitly, and visualize the state transition graph. The GridWorld you built here will become a concrete MDP.

## Additional Resources

- Sutton & Barto, *Reinforcement Learning: An Introduction*, Chapter 1 (free at incompleteideas.net)
- The GridWorld interface intentionally mirrors the OpenAI Gymnasium API — the same `reset`/`step` pattern works for Atari games, robotic arms, and financial simulators

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])